In [ ]:
!pip install ultralytics==8.3.19
!pip install supervision[assets]==0.24.0

import os
import cv2
import numpy as np
import supervision as sv
import ultralytics
import matplotlib.pyplot as plt
from ultralytics import YOLO
from IPython import display

display.clear_output()
ultralytics.checks()

display.clear_output()
print('supervision.__version:', sv.__version__)

HOME = os.getcwd()
print(HOME)

In [ ]:
from google.colab import files

uploaded = files.upload()

SOURCE_VIDEO_PATH = list(uploaded.keys())[0]
print("🎬 Video đầu vào:", SOURCE_VIDEO_PATH)

In [ ]:
model = YOLO("yolo11x.pt")

CLASS_NAMES_DICT = model.model.names

SELECTED_CLASS_NAMES = ['car', 'bus', 'truck', 'motorcycle']
SELECTED_CLASS_IDS = [
    {value: key for key, value in CLASS_NAMES_DICT.items()}[class_name]
    for class_name in SELECTED_CLASS_NAMES
]

In [ ]:
generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)

box_annotator = sv.BoxAnnotator(thickness=4)
label_annotator = sv.LabelAnnotator(text_thickness=2, text_scale=1.5, text_color=sv.Color.BLACK)

iterator = iter(generator)
frame = next(iterator)

results = model(frame, verbose=False)[0]
detections = sv.Detections.from_ultralytics(results)

detections = detections[np.isin(detections.class_id, SELECTED_CLASS_IDS)]

labels = [
    f"{CLASS_NAMES_DICT[class_id]} {confidence:0.2f}"
    for confidence, class_id in zip(detections.confidence, detections.class_id)
]

annotated_frame = frame.copy()
annotated_frame = box_annotator.annotate(scene=annotated_frame, detections=detections)
annotated_frame = label_annotator.annotate(scene=annotated_frame, detections=detections, labels=labels)

sv.plot_image(annotated_frame, (16, 16))

In [ ]:
generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)
iterator = iter(generator)
frame = next(iterator)

print("Frame shape (h, w, c):", frame.shape)

plt.figure(figsize=(10, 6))
plt.imshow(frame[:, :, ::-1])
plt.axis('on')

## **ByteTrack**

In [ ]:
LINE_START = sv.Point(5, 500)
LINE_END   = sv.Point(1915, 500)
TARGET_VIDEO_PATH = f"{HOME}/result.mp4"

sv.VideoInfo.from_video_path(SOURCE_VIDEO_PATH)

byte_tracker = sv.ByteTrack(
    track_activation_threshold=0.25,
    lost_track_buffer=30,
    minimum_matching_threshold=0.8,
    frame_rate=30,
    minimum_consecutive_frames=3
)

byte_tracker.reset()

video_info = sv.VideoInfo.from_video_path(SOURCE_VIDEO_PATH)
generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)

line_zone = sv.LineZone(start=LINE_START, end=LINE_END)

line_y = LINE_START.y
previous_positions = {}
class_counts = {name: 0 for name in SELECTED_CLASS_NAMES}
crossed_ids = set()

box_annotator = sv.BoxAnnotator(thickness=4)
label_annotator = sv.LabelAnnotator(text_thickness=2, text_scale=1.5, text_color=sv.Color.BLACK)
trace_annotator = sv.TraceAnnotator(thickness=4, trace_length=50)
line_zone_annotator = sv.LineZoneAnnotator(thickness=4, color=sv.Color.RED, text_thickness=2, text_scale=2, display_in_count=False, display_out_count=False)

def callback(frame: np.ndarray, index: int) -> np.ndarray:
    global previous_positions, class_counts, crossed_ids

    results = model(frame, verbose=False)[0]
    detections = sv.Detections.from_ultralytics(results)

    detections = detections[np.isin(detections.class_id, SELECTED_CLASS_IDS)]
    detections = byte_tracker.update_with_detections(detections)

    if detections.tracker_id is not None:
        xyxy = detections.xyxy
        for i in range(len(detections)):
            tid = int(detections.tracker_id[i])
            cls_id = int(detections.class_id[i])
            cls_name = CLASS_NAMES_DICT[cls_id]
            cx = (xyxy[i, 0] + xyxy[i, 2]) / 2
            cy = (xyxy[i, 1] + xyxy[i, 3]) / 2

            if tid in previous_positions:
                py = previous_positions[tid]
                if py < line_y and cy > line_y and (tid, 'out') not in crossed_ids:
                    crossed_ids.add((tid, 'out'))
                    class_counts[cls_name] = class_counts.get(cls_name, 0) + 1
                elif py > line_y and cy < line_y and (tid, 'in') not in crossed_ids:
                    crossed_ids.add((tid, 'in'))
                    class_counts[cls_name] = class_counts.get(cls_name, 0) + 1
            previous_positions[tid] = cy

    labels = [
        f"#{tracker_id} {model.model.names[class_id]} {confidence:0.2f}"
        for confidence, class_id, tracker_id in zip(
            detections.confidence, detections.class_id, detections.tracker_id
        )
    ]

    annotator_frame = frame.copy()
    annotator_frame = trace_annotator.annotate(scene=annotator_frame, detections=detections)
    annotator_frame = box_annotator.annotate(scene=annotator_frame, detections=detections)
    annotator_frame = label_annotator.annotate(scene=annotator_frame, detections=detections, labels=labels)

    line_zone.trigger(detections)
    annotator_frame = line_zone_annotator.annotate(annotator_frame, line_counter=line_zone)

    h, w, _ = annotator_frame.shape
    box_w, box_h = 280, 50 + len(SELECTED_CLASS_NAMES) * 28
    x0, y0 = w - box_w - 20, 20

    overlay = annotator_frame.copy()
    cv2.rectangle(overlay, (x0, y0), (x0 + box_w, y0 + box_h), (0, 0, 0), -1)
    annotator_frame = cv2.addWeighted(overlay, 0.6, annotator_frame, 0.4, 0)

    total = sum(class_counts.values())
    cv2.putText(annotator_frame, f'Total: {total}', (x0 + 10, y0 + 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
    for i, cls_name in enumerate(SELECTED_CLASS_NAMES):
        cnt = class_counts.get(cls_name, 0)
        cv2.putText(annotator_frame, f'{cls_name.capitalize()}: {cnt}',
                    (x0 + 10, y0 + 60 + i * 28),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    return annotator_frame

sv.process_video(
    source_path=SOURCE_VIDEO_PATH,
    target_path=TARGET_VIDEO_PATH,
    callback=callback
)

In [ ]:
from google.colab import files

files.download('result.mp4')